# 🚀 Training MG-MMKT on Kaggle (2x NVIDIA Tesla T4 GPUs)
**Mô hình:** Masked-Gated Multimodal Knowledge Tracing (MG-MMKT)  
**Tập dữ liệu:** Preprocessed XES3G5M (Chinese Mathematics Multimodal Knowledge Tracing - 5,549,184 tương tác)  
**Mục tiêu:** Tận dụng 2x GPU NVIDIA Tesla T4 (mỗi GPU 16GB VRAM) để huấn luyện tối ưu tốc độ, hỗ trợ chạy song song toàn diện **5-Fold Cross-Validation cho Full Model và TẤT CẢ các khâu thực nghiệm loại suy (Ablation Studies)** cùng mô hình cơ sở Baseline ID-Only SAKT.

---

### ⚙️ Hướng Dẫn Cấu Hình Kaggle Notebook (Bắt Buộc):
1. **Accelerator:** Ở thanh menu bên phải (**Session options**), chọn **GPU T4 x 2**.
2. **Internet:** Chuyển sang **Internet ON** (bắt buộc để cài package và clone/pull git repository).
3. **Data Input:** Nhấn **+ Add Input**, chọn Dataset XES3G5M đã nén/tiền xử lý của bạn (được gắn vào thư mục `/kaggle/input/...`).

In [ ]:
# [1] Kiểm tra cấu hình phần cứng & Cấu hình chống phân mảnh VRAM
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi

import torch
print(f"🔥 PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"Số lượng GPU nhận diện: {gpu_count}")

for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | Compute Capability: {props.major}.{props.minor}")

assert gpu_count >= 1, "⚠️ Vui lòng chuyển Accelerator sang GPU T4 x 2 trong cài đặt Kaggle!"

In [ ]:
# [2] Thiết lập môi trường làm việc & Clone/Pull repository từ GitHub
import os
import sys

REPO_URL = "https://github.com/nbao0208/kt-mmkt-research.git"
REPO_DIR = "/kaggle/working/kt-mmkt-research"

if os.path.exists("src/models/mmkt/mg_mmkt.py"):
    print("✅ Bạn đang đứng trực tiếp trong thư mục repository kt-mmkt-research.")
    !git pull origin main
elif os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"✅ Đã chuyển thư mục làm việc đến: {REPO_DIR}")
    !git pull origin main
else:
    print(f"📥 Đang clone repository từ {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    print(f"✅ Đã clone thành công và chuyển vào: {os.getcwd()}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working Directory:", os.getcwd())

In [ ]:
# [3] Cài đặt & cập nhật các thư viện phụ thuộc (Dependencies)
!pip install -q -r requirements.txt
!pip install -q seaborn matplotlib scikit-learn tabulate
print("✅ Hoàn tất cài đặt toàn bộ gói phụ thuộc!")

In [ ]:
# [4] Tự động phát hiện và liên kết (Symlink) tập dữ liệu XES3G5M từ /kaggle/input/
import glob
import torch
from pathlib import Path

# 1. Tìm thư mục chứa file static item bank hoặc sequence files
candidates = glob.glob("/kaggle/input/**/sequences_fold_0.pt", recursive=True)
if not candidates:
    candidates = glob.glob("/kaggle/input/**/static_item_bank.pt", recursive=True)
if not candidates:
    candidates = glob.glob("/kaggle/input/**/mg_static_item_bank.pt", recursive=True)

if not candidates:
    raise FileNotFoundError(
        "❌ Không tìm thấy tập dữ liệu XES3G5M trong /kaggle/input/!\n"
        "👉 Vui lòng nhấn nút '+ Add Input' trên giao diện Kaggle và thêm Dataset XES3G5M đã tiền xử lý của bạn."
    )

input_data_dir = Path(candidates[0]).parent
print(f"🔍 Đã tìm thấy thư mục chứa dữ liệu tại: {input_data_dir}")

target_dir = Path("data/processed/XES3G5M")
target_dir.mkdir(parents=True, exist_ok=True)

# 2. Tạo symlink cho toàn bộ file .pt và .json
linked_count = 0
for ext in ["*.pt", "*.json"]:
    for src_file in input_data_dir.glob(ext):
        dst_file = target_dir / src_file.name
        if dst_file.exists() or dst_file.is_symlink():
            dst_file.unlink()
        dst_file.symlink_to(src_file)
        linked_count += 1

print(f"✅ Đã tạo symbolic link cho {linked_count} tệp dữ liệu vào {target_dir}.")

# 3. Tối ưu hóa: Nếu có static_item_bank.pt (4.6GB) nhưng chưa có mg_static_item_bank.pt (~48MB),
# tự động trích xuất để nạp siêu nhanh và tiết kiệm RAM tối đa cho MG-MMKT!
from src.data.xes3g5m_dataset import create_mg_static_item_bank

full_bank_path = target_dir / "static_item_bank.pt"
mg_bank_path = target_dir / "mg_static_item_bank.pt"

if not mg_bank_path.exists() and full_bank_path.exists():
    print("⚡ Đang trích xuất mg_static_item_bank.pt nhẹ (~48 MB) từ file 4.6 GB...")
    create_mg_static_item_bank(str(full_bank_path), str(mg_bank_path))
    print("✅ Trích xuất hoàn tất!")

print("\n📁 Danh sách các tệp dữ liệu sẵn sàng cho huấn luyện:")
!ls -lh data/processed/XES3G5M

In [ ]:
# [5] Chạy Sanity Check Unit Tests để đảm bảo kiến trúc MG-MMKT và CUDA hoạt động hoàn hảo
!python -m pytest tests/test_mg_mmkt_model.py -v
print("🎉 Toàn bộ Unit Tests cho kiến trúc MG-MMKT đã vượt qua thành công!")

## 🧠 Kiến Trúc MG-MMKT & Thiết Kế Các Khâu Loại Suy (Ablation Studies)

Theo tài liệu đặc tả kiến trúc `docs/models/MG-MMKT.md`, mô hình **MG-MMKT** giải quyết 2 ràng buộc cấu trúc cốt lõi trong dữ liệu giáo dục:
1. **Tính khuyết thiếu phương thức có cấu trúc (Structural Modality Incompleteness):** Xử lý bằng Vector Đại diện Khuyết thiếu Học được ($e_{\text{null}}^{\text{img}, q}, e_{\text{null}}^{\text{txt}, s}, e_{\text{null}}^{\text{img}, s}$) và Cổng Softmax Thích ứng Có Mặt nạ (Masked Softmax Gating).
2. **Sự bất đối xứng nhân quả thời gian của lời giải (Temporal-Causal Asymmetry of Solutions):** Tách biệt tương tác lịch sử $y_\tau$ (học sinh đã thấy lời giải sau phản hồi $r_\tau$) và truy vấn mục tiêu $x_t$ (chỉ chứa đề bài, hoàn toàn không làm rò rỉ lời giải $s_t$).

### 🔬 Bảng Cấu Hình Các Khâu Loại Suy (Ablation Matrix):
| Cấu hình | Tên Thực Nghiệm | Cờ CLI | Mô Tả Kỹ Thuật | Mục Đích Đánh Giá |
| :--- | :--- | :--- | :--- | :--- |
| **Config 1** | **Full MG-MMKT** | `--ablation 1` | Đầy đủ: ID + Text + Visual + Solution + Learnable $e_{\text{null}}$ + Masked Gated Fusion | Đường cơ sở tối ưu toàn diện của kiến trúc đề xuất |
| **Config 2** | **w/o Solution** | `--ablation 2` | Loại bỏ hoàn toàn biểu diễn lời giải trong lịch sử ($z_\tau^s$) | Kiểm chứng giá trị can thiệp sư phạm hậu nghiệm của lời giải |
| **Config 3** | **w/o Visual** | `--ablation 3` | Loại bỏ nhánh ảnh của đề bài và lời giải ($v_{\text{img}}^q, v_{\text{img}}^s$) | Đánh giá mức độ cần thiết của thông tin thị giác |
| **Config 4** | **Naive Concat** | `--ablation 4` | Zero-padding khi khuyết phương thức + Phép nối tuyến tính (Linear Concat), bỏ Softmax Gate | Chứng minh hiện tượng sụp đổ chuẩn vector và hiệu quả của cơ chế Gated Fusion |
| **Baseline** | **ID-Only SAKT** | `--model-type sakt` | Chỉ sử dụng ID câu hỏi và ID kỹ năng (SAKT truyền thống) | Đo lường mức cải thiện vượt bậc của MMKT so với KT truyền thống |

### ⚡ Chiến Lược Tận Dụng 2x GPU Tesla T4:
- **Song song hóa 5-Fold Cross-Validation:** Với mỗi cấu hình, GPU 0 đảm nhận Folds [0, 1, 2], GPU 1 đảm nhận Folds [3, 4] song song không độ trễ, hoàn thành 5 fold chỉ trong ~8-10 phút.
- **Lựa chọn chế độ thực nghiệm:** Bạn có thể bật `RUN_5FOLD_ABLATION = True` để chạy 5-Fold cho toàn bộ các khâu ablation (~40 phút), hoặc `RUN_5FOLD_ABLATION = False` để chạy nhanh trên Fold 0 (~10 phút).

In [ ]:
# ==============================================================================
# [Cấu Hình Huấn Luyện Toàn Cục] Thiết lập chế độ chạy 5-Fold cho các khâu Ablation
# ==============================================================================
# 👉 Đặt RUN_5FOLD_ABLATION = True để huấn luyện đầy đủ 5-Fold cho TẤT CẢ các khâu Ablation!
#    (Thời gian: ~40-45 phút trên 2x T4, xuất kết quả Mean ± Std chuẩn bài báo khoa học)
# 👉 Đặt RUN_5FOLD_ABLATION = False để chạy nhanh trên Fold 0 (~10 phút kiểm tra nhanh)
RUN_5FOLD_ABLATION = True

NUM_EPOCHS = 30
BATCH_SIZE = 64
LR = 1e-3

print(f"⚙️ Chế độ huấn luyện Ablation: {'5-FOLD CROSS-VALIDATION' if RUN_5FOLD_ABLATION else 'SINGLE-FOLD RAPID (Fold 0)'}")
print(f"   Epochs: {NUM_EPOCHS} | Batch Size: {BATCH_SIZE} | LR: {LR}")

In [ ]:
# ==============================================================================
# [Kịch bản 1] Huấn luyện Mô hình Gốc Full MG-MMKT (Config 1)
# ==============================================================================
# - Chạy 5-Fold Cross-Validation song song trên 2x GPU T4 (GPU 0: Folds 0,1,2 | GPU 1: Folds 3,4)
# - Tự động tổng hợp bảng kết quả 5-Fold (Mean ± Std) chuẩn pyKT

os.makedirs("outputs/checkpoints/full_5fold", exist_ok=True)
os.makedirs("outputs/reports/full_5fold", exist_ok=True)

!python -u scripts/run_5fold_mg_mmkt.py \
    --dual-gpu \
    --ablation 1 \
    --num-epochs {NUM_EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} \
    --save-base-dir outputs/checkpoints/full_5fold \
    --output-report-dir outputs/reports/full_5fold

## 🧪 Thực Hiện Toàn Bộ Các Khâu Loại Suy (Ablation Studies)

Các ô lệnh dưới đây tự động thích ứng theo biến `RUN_5FOLD_ABLATION`:
- Nếu `RUN_5FOLD_ABLATION = True`: Chạy đầy đủ 5-Fold song song trên 2x GPU T4 cho từng cấu hình loại suy (Ablation 2, 3, 4 và Baseline SAKT).
- Nếu `RUN_5FOLD_ABLATION = False`: Chạy đơn Fold (Fold 0) song song theo cặp trên 2 GPU.

In [ ]:
# ==============================================================================
# [Ablation Đợt 1] Huấn luyện Ablation 2 (w/o Solution) & Ablation 3 (w/o Visual)
# ==============================================================================
import subprocess
import os
import sys
import time

os.makedirs("outputs/logs", exist_ok=True)

if RUN_5FOLD_ABLATION:
    print("🚀 [1/2] Đang huấn luyện 5-Fold cho Ablation 2 (w/o Solution) trên 2x GPU T4...")
    !python -u scripts/run_5fold_mg_mmkt.py \
        --dual-gpu \
        --ablation 2 \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --save-base-dir outputs/checkpoints/ablation2_5fold \
        --output-report-dir outputs/reports/ablation2_5fold

    print("\n🚀 [2/2] Đang huấn luyện 5-Fold cho Ablation 3 (w/o Visual) trên 2x GPU T4...")
    !python -u scripts/run_5fold_mg_mmkt.py \
        --dual-gpu \
        --ablation 3 \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --save-base-dir outputs/checkpoints/ablation3_5fold \
        --output-report-dir outputs/reports/ablation3_5fold
else:
    # Chế độ chạy nhanh Fold 0 theo cặp song song
    log_abl2 = "outputs/logs/ablation2_gpu0.log"
    log_abl3 = "outputs/logs/ablation3_gpu1.log"
    env_gpu0 = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTHONUNBUFFERED": "1"}
    env_gpu1 = {**os.environ, "CUDA_VISIBLE_DEVICES": "1", "PYTHONUNBUFFERED": "1"}

    print("🚀 Chạy song song Fold 0: Ablation 2 (GPU 0) vs Ablation 3 (GPU 1)...")
    cmd_abl3 = [sys.executable, "-u", "scripts/train_mg_mmkt.py", "--val-fold", "0", "--ablation", "3", "--device", "cuda:0", "--batch-size", str(BATCH_SIZE), "--num-epochs", str(NUM_EPOCHS), "--save-dir", "outputs/checkpoints/ablation3_wo_visual"]
    f_abl3 = open(log_abl3, "w")
    p_abl3 = subprocess.Popen(cmd_abl3, stdout=f_abl3, stderr=subprocess.STDOUT, env=env_gpu1)

    time.sleep(5)
    cmd_abl2 = [sys.executable, "-u", "scripts/train_mg_mmkt.py", "--val-fold", "0", "--ablation", "2", "--device", "cuda:0", "--batch-size", str(BATCH_SIZE), "--num-epochs", str(NUM_EPOCHS), "--save-dir", "outputs/checkpoints/ablation2_wo_solution"]
    f_abl2 = open(log_abl2, "w")
    p_abl2 = subprocess.Popen(cmd_abl2, stdout=f_abl2, stderr=subprocess.STDOUT, env=env_gpu0)

    p_abl2.wait(); f_abl2.close()
    p_abl3.wait(); f_abl3.close()
    print("✅ Hoàn thành Đợt 1 (Fold 0)!")

In [ ]:
# ==============================================================================
# [Ablation Đợt 2] Huấn luyện Ablation 4 (Naive Concat) & Baseline ID-Only SAKT
# ==============================================================================
if RUN_5FOLD_ABLATION:
    print("🚀 [1/2] Đang huấn luyện 5-Fold cho Ablation 4 (Naive Concat) trên 2x GPU T4...")
    !python -u scripts/run_5fold_mg_mmkt.py \
        --dual-gpu \
        --ablation 4 \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --save-base-dir outputs/checkpoints/ablation4_5fold \
        --output-report-dir outputs/reports/ablation4_5fold

    print("\n🚀 [2/2] Đang huấn luyện 5-Fold cho Baseline (ID-Only SAKT) trên 2x GPU T4...")
    !python -u scripts/run_5fold_mg_mmkt.py \
        --dual-gpu \
        --model-type sakt \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --save-base-dir outputs/checkpoints/baseline_sakt_5fold \
        --output-report-dir outputs/reports/baseline_sakt_5fold
else:
    # Chế độ chạy nhanh Fold 0 theo cặp song song
    log_abl4 = "outputs/logs/ablation4_gpu0.log"
    log_sakt = "outputs/logs/baseline_sakt_gpu1.log"
    env_gpu0 = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTHONUNBUFFERED": "1"}
    env_gpu1 = {**os.environ, "CUDA_VISIBLE_DEVICES": "1", "PYTHONUNBUFFERED": "1"}

    print("🚀 Chạy song song Fold 0: Ablation 4 (GPU 0) vs Baseline SAKT (GPU 1)...")
    cmd_sakt = [sys.executable, "-u", "scripts/run_5fold_mg_mmkt.py", "--folds", "0", "--model-type", "sakt", "--device", "cuda:0", "--batch-size", str(BATCH_SIZE), "--num-epochs", str(NUM_EPOCHS), "--save-base-dir", "outputs/checkpoints/baseline_sakt", "--output-report-dir", "outputs/reports/baseline_sakt", "--worker-only"]
    f_sakt = open(log_sakt, "w")
    p_sakt = subprocess.Popen(cmd_sakt, stdout=f_sakt, stderr=subprocess.STDOUT, env=env_gpu1)

    time.sleep(5)
    cmd_abl4 = [sys.executable, "-u", "scripts/train_mg_mmkt.py", "--val-fold", "0", "--ablation", "4", "--device", "cuda:0", "--batch-size", str(BATCH_SIZE), "--num-epochs", str(NUM_EPOCHS), "--save-dir", "outputs/checkpoints/ablation4_naive_concat"]
    f_abl4 = open(log_abl4, "w")
    p_abl4 = subprocess.Popen(cmd_abl4, stdout=f_abl4, stderr=subprocess.STDOUT, env=env_gpu0)

    p_abl4.wait(); f_abl4.close()
    p_sakt.wait(); f_sakt.close()
    print("✅ Hoàn thành Đợt 2 (Fold 0)!")

In [ ]:
# [6] Đánh giá chi tiết tập Test (1.102.810 tương tác) & Báo cáo Khả năng Giải thích Bậc 1
import os

os.makedirs("outputs/reports", exist_ok=True)

# Đánh giá chi tiết trên checkpoint Fold 0 để trích xuất phân tích đa phương thức & giải thích
if RUN_5FOLD_ABLATION:
    ckpt_full = "outputs/checkpoints/full_5fold/mg_mmkt_fold0/best_model.pt"
    ckpt_abl2 = "outputs/checkpoints/ablation2_5fold/mg_mmkt_fold0/best_model.pt"
    ckpt_abl3 = "outputs/checkpoints/ablation3_5fold/mg_mmkt_fold0/best_model.pt"
    ckpt_abl4 = "outputs/checkpoints/ablation4_5fold/mg_mmkt_fold0/best_model.pt"
else:
    ckpt_full = "outputs/checkpoints/mg_mmkt_fold0/best_model.pt"
    ckpt_abl2 = "outputs/checkpoints/ablation2_wo_solution/best_model.pt"
    ckpt_abl3 = "outputs/checkpoints/ablation3_wo_visual/best_model.pt"
    ckpt_abl4 = "outputs/checkpoints/ablation4_naive_concat/best_model.pt"

eval_list = [
    ("Full MG-MMKT", ckpt_full, "outputs/reports/eval_full_model.json"),
    ("Ablation 2 (w/o Solution)", ckpt_abl2, "outputs/reports/eval_ablation2.json"),
    ("Ablation 3 (w/o Visual)", ckpt_abl3, "outputs/reports/eval_ablation3.json"),
    ("Ablation 4 (Naive Concat)", ckpt_abl4, "outputs/reports/eval_ablation4.json"),
]

for name, ckpt_path, rep_path in eval_list:
    if os.path.exists(ckpt_path):
        print(f"\n🔍 Đang đánh giá phân tích chi tiết cho {name} (Fold 0)...\n")
        !python -u scripts/evaluate_mg_mmkt.py \
            --checkpoint {ckpt_path} \
            --val-fold 0 \
            --device cuda:0 \
            --output-report {rep_path}
    else:
        print(f"⚠️ Bỏ qua {name} (Chưa tìm thấy checkpoint: {ckpt_path})")

In [ ]:
# [7] Bảng Tổng Hợp So Sánh Toàn Bộ Các Mô Hình & Kiểm Chứng Giả Thuyết Khoa Học
import json
import os
import pandas as pd

results = []

if RUN_5FOLD_ABLATION:
    # Nạp báo cáo tổng hợp 5-Fold (Mean ± Std) từ outputs/reports/
    configs_5fold = [
        ("Full MG-MMKT (Config 1)", "outputs/reports/full_5fold/5fold_summary_abl1.json", "outputs/reports/full_5fold/5fold_summary.json", "Mô hình hoàn chỉnh đề xuất"),
        ("Ablation 2 (w/o Solution)", "outputs/reports/ablation2_5fold/5fold_summary_abl2.json", "outputs/reports/ablation2_5fold/5fold_summary.json", "Loại bỏ lời giải quá khứ"),
        ("Ablation 3 (w/o Visual)", "outputs/reports/ablation3_5fold/5fold_summary_abl3.json", "outputs/reports/ablation3_5fold/5fold_summary.json", "Loại bỏ thông tin hình ảnh"),
        ("Ablation 4 (Naive Concat)", "outputs/reports/ablation4_5fold/5fold_summary_abl4.json", "outputs/reports/ablation4_5fold/5fold_summary.json", "Zero-imputation + Tuyến tính nối"),
        ("Baseline (ID-Only SAKT)", "outputs/reports/baseline_sakt_5fold/5fold_summary.json", None, "SAKT đơn phương thức (ID thuần)"),
    ]

    full_mean_auc = None
    for name, p1, p2, desc in configs_5fold:
        path = p1 if (p1 and os.path.exists(p1)) else (p2 if (p2 and os.path.exists(p2)) else None)
        if path:
            with open(path) as f:
                d = json.load(f)
            mean_auc = d.get("mean_test_auc", 0.0)
            std_auc = d.get("std_test_auc", 0.0)
            mean_acc = d.get("mean_test_acc", 0.0)
            std_acc = d.get("std_test_acc", 0.0)
            if full_mean_auc is None and "Full" in name:
                full_mean_auc = mean_auc
            delta = f"{mean_auc - full_mean_auc:+.4f}" if full_mean_auc else "--"
            results.append({
                "Model Variant": name,
                "5-Fold Test AUC": f"{mean_auc:.4f} ± {std_auc:.4f}",
                "5-Fold Test ACC": f"{mean_acc:.4f} ± {std_acc:.4f}",
                "Δ AUC (vs Full)": delta,
                "Description": desc
            })

    df_results = pd.DataFrame(results)
    print("\n🏆 BẢNG TỔNG HỢP SO SÁNH 5-FOLD CROSS-VALIDATION (MEAN ± STD) TRÊN XES3G5M:")
    display(df_results)
    df_results.to_markdown("outputs/reports/ablation_synthesis_table.md", index=False)
    df_results.to_json("outputs/reports/ablation_synthesis_table.json", indent=2, orient="records")

else:
    # Bảng tổng hợp Single-Fold (Fold 0)
    report_configs = [
        ("Full MG-MMKT (Config 1)", "outputs/reports/eval_full_model.json", "Mô hình hoàn chỉnh đề xuất"),
        ("Ablation 2 (w/o Solution)", "outputs/reports/eval_ablation2.json", "Loại bỏ lời giải quá khứ"),
        ("Ablation 3 (w/o Visual)", "outputs/reports/eval_ablation3.json", "Loại bỏ thông tin hình ảnh"),
        ("Ablation 4 (Naive Concat)", "outputs/reports/eval_ablation4.json", "Zero-imputation + Tuyến tính nối"),
    ]
    full_auc = None
    for name, path, desc in report_configs:
        if os.path.exists(path):
            with open(path) as f:
                d = json.load(f)
            sm = d.get("stratified_metrics", {})
            ov = sm.get("overall", {})
            img_q = sm.get("questions_with_image", {})
            txt_q = sm.get("questions_text_only", {})
            auc = ov.get("auc", 0.0)
            acc = ov.get("acc", 0.0)
            if full_auc is None:
                full_auc = auc
            results.append({
                "Model Variant": name,
                "Overall Test AUC": f"{auc:.4f}",
                "Overall Test ACC": f"{acc:.4f}",
                "Δ AUC (vs Full)": f"{auc - full_auc:+.4f}" if full_auc else "--",
                "With-Image AUC": f"{img_q.get('auc', 0.0):.4f}",
                "Text-Only AUC": f"{txt_q.get('auc', 0.0):.4f}",
                "Description": desc
            })
    df_results = pd.DataFrame(results)
    print("\n🏆 BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG TẬP TEST (FOLD 0) TRÊN XES3G5M:")
    display(df_results)
    df_results.to_markdown("outputs/reports/ablation_synthesis_table.md", index=False)
    df_results.to_json("outputs/reports/ablation_synthesis_table.json", indent=2, orient="records")

print("\n✅ Báo cáo so sánh đã được lưu tại outputs/reports/ablation_synthesis_table.md")

In [ ]:
# [8] Trực quan hóa Biểu Đồ Nghiên Cứu Chất Lượng Cao (Publication-Ready Figures)
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

os.makedirs("outputs/reports/figures", exist_ok=True)
sns.set_theme(style="whitegrid", font_scale=1.05)

fig = plt.figure(figsize=(18, 5.5))
gs = fig.add_gridspec(1, 3, wspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])

# -------------------------------------------------------------------------
# (a) Đồ thị tiến trình Validation AUC theo Epochs
# -------------------------------------------------------------------------
if RUN_5FOLD_ABLATION:
    models_history = [
        ("Full MG-MMKT", "outputs/checkpoints/full_5fold/mg_mmkt_fold0/training_summary.json", "#1f77b4"),
        ("w/o Solution", "outputs/checkpoints/ablation2_5fold/mg_mmkt_fold0/training_summary.json", "#ff7f0e"),
        ("w/o Visual", "outputs/checkpoints/ablation3_5fold/mg_mmkt_fold0/training_summary.json", "#2ca02c"),
        ("Naive Concat", "outputs/checkpoints/ablation4_5fold/mg_mmkt_fold0/training_summary.json", "#d62728"),
    ]
else:
    models_history = [
        ("Full MG-MMKT", "outputs/checkpoints/mg_mmkt_fold0/training_summary.json", "#1f77b4"),
        ("w/o Solution", "outputs/checkpoints/ablation2_wo_solution/training_summary.json", "#ff7f0e"),
        ("w/o Visual", "outputs/checkpoints/ablation3_wo_visual/training_summary.json", "#2ca02c"),
        ("Naive Concat", "outputs/checkpoints/ablation4_naive_concat/training_summary.json", "#d62728"),
    ]

plotted_any = False
for label, path, color in models_history:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        history = data.get("history", [])
        epochs = [h.get("epoch", i+1) for i, h in enumerate(history)]
        val_aucs = [h.get("val_auc", 0) for h in history]
        ax1.plot(epochs, val_aucs, label=label, color=color, linewidth=2.0, marker="o", markersize=3.5)
        plotted_any = True

if plotted_any:
    ax1.set_title("(a) Validation ROC-AUC Progression (Fold 0)", fontsize=12, fontweight="bold", pad=10)
    ax1.set_xlabel("Epoch", fontsize=11)
    ax1.set_ylabel("Validation ROC-AUC", fontsize=11)
    ax1.legend(loc="lower right", frameon=True, fontsize=9.5)
    ax1.grid(True, linestyle="--", alpha=0.6)

# -------------------------------------------------------------------------
# (b) Đồ thị Cột So Sánh Test ROC-AUC giữa các cấu hình Loại suy
# -------------------------------------------------------------------------
if RUN_5FOLD_ABLATION:
    report_files = [
        ("Full MG-MMKT", "outputs/reports/full_5fold/5fold_summary_abl1.json", "outputs/reports/full_5fold/5fold_summary.json"),
        ("w/o Solution", "outputs/reports/ablation2_5fold/5fold_summary_abl2.json", "outputs/reports/ablation2_5fold/5fold_summary.json"),
        ("w/o Visual", "outputs/reports/ablation3_5fold/5fold_summary_abl3.json", "outputs/reports/ablation3_5fold/5fold_summary.json"),
        ("Naive Concat", "outputs/reports/ablation4_5fold/5fold_summary_abl4.json", "outputs/reports/ablation4_5fold/5fold_summary.json"),
    ]
    model_names, test_aucs, std_aucs = [], [], []
    for name, p1, p2 in report_files:
        path = p1 if os.path.exists(p1) else (p2 if (p2 and os.path.exists(p2)) else None)
        if path:
            with open(path) as f:
                d = json.load(f)
            model_names.append(name)
            test_aucs.append(d.get("mean_test_auc", 0.0))
            std_aucs.append(d.get("std_test_auc", 0.0))
    if test_aucs:
        palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
        bars = ax2.bar(model_names, test_aucs, yerr=std_aucs, capsize=5, color=palette[:len(model_names)], width=0.55, edgecolor="black", linewidth=0.8)
        ax2.set_title("(b) 5-Fold Test ROC-AUC (Mean ± Std)", fontsize=12, fontweight="bold", pad=10)
        ax2.set_ylabel("Test ROC-AUC", fontsize=11)
        ax2.set_ylim(max(0.5, min(test_aucs) - 0.02), max(test_aucs) + 0.02)
        for bar, auc_val, std_val in zip(bars, test_aucs, std_aucs):
            yval = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2.0, yval + std_val + 0.002, f"{auc_val:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=10)
        ax2.tick_params(axis="x", rotation=15)
        ax2.grid(axis="y", linestyle="--", alpha=0.6)
else:
    report_configs = [
        ("Full MG-MMKT", "outputs/reports/eval_full_model.json"),
        ("w/o Solution", "outputs/reports/eval_ablation2.json"),
        ("w/o Visual", "outputs/reports/eval_ablation3.json"),
        ("Naive Concat", "outputs/reports/eval_ablation4.json"),
    ]
    model_names, test_aucs = [], []
    for name, path in report_configs:
        if os.path.exists(path):
            with open(path) as f:
                rep = json.load(f)
            model_names.append(name)
            test_aucs.append(rep.get("stratified_metrics", {}).get("overall", {}).get("auc", 0.0))
    if test_aucs:
        palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
        bars = ax2.bar(model_names, test_aucs, color=palette[:len(model_names)], width=0.55, edgecolor="black", linewidth=0.8)
        ax2.set_title("(b) Test ROC-AUC Comparison (Fold 0)", fontsize=12, fontweight="bold", pad=10)
        ax2.set_ylabel("Test ROC-AUC", fontsize=11)
        ax2.set_ylim(max(0.5, min(test_aucs) - 0.02), max(test_aucs) + 0.02)
        for bar in bars:
            yval = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2.0, yval + 0.002, f"{yval:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=10)
        ax2.tick_params(axis="x", rotation=15)
        ax2.grid(axis="y", linestyle="--", alpha=0.6)

# -------------------------------------------------------------------------
# (c) Đồ thị Đóng Góp Phương Thức (Modality Attribution Weights alpha)
# -------------------------------------------------------------------------
full_report_path = "outputs/reports/eval_full_model.json"
if os.path.exists(full_report_path):
    with open(full_report_path) as f:
        full_d = json.load(f)
    mod_attr = full_d.get("question_modality_attribution", {})
    w_img = mod_attr.get("with_image", {})
    txt_o = mod_attr.get("text_only", {})

    cats = ["With-Image Qs", "Text-Only Qs"]
    alpha_ids = [w_img.get("mean_alpha_id", 0.33), txt_o.get("mean_alpha_id", 0.45)]
    alpha_txts = [w_img.get("mean_alpha_txt", 0.33), txt_o.get("mean_alpha_txt", 0.55)]
    alpha_imgs = [w_img.get("mean_alpha_img", 0.34), txt_o.get("mean_alpha_img", 0.00)]

    x = np.arange(len(cats))
    width = 0.25
    ax3.bar(x - width, alpha_ids, width, label="Alpha ID", color="#4e79a7", edgecolor="black", linewidth=0.8)
    ax3.bar(x, alpha_txts, width, label="Alpha Text", color="#59a14f", edgecolor="black", linewidth=0.8)
    ax3.bar(x + width, alpha_imgs, width, label="Alpha Image", color="#e15759", edgecolor="black", linewidth=0.8)

    ax3.set_title("(c) Gated Modality Attribution (Alpha Weights)", fontsize=12, fontweight="bold", pad=10)
    ax3.set_xticks(x)
    ax3.set_xticklabels(cats, fontsize=10.5)
    ax3.set_ylabel("Attribution Weight", fontsize=11)
    ax3.set_ylim(0.0, 1.0)
    ax3.legend(loc="upper right", frameon=True, fontsize=9.5)
    ax3.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
fig_path = "outputs/reports/figures/mg_mmkt_comprehensive_analysis.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Toàn bộ biểu đồ nghiên cứu đã được lưu tại: {fig_path}")

In [ ]:
# [9] Đóng gói toàn bộ Checkpoints, Reports, Logs và Figures thành file ZIP để tải về
!zip -q -r /kaggle/working/mg_mmkt_results.zip outputs/checkpoints outputs/reports outputs/logs
zip_size_mb = os.path.getsize("/kaggle/working/mg_mmkt_results.zip") / (1024**2)
print(f"🎉 Đã nén thành công toàn bộ kết quả vào: /kaggle/working/mg_mmkt_results.zip ({zip_size_mb:.2f} MB)")
print("👉 Bạn có thể tải file này trực tiếp từ khung 'Output' bên phải màn hình giao diện Kaggle!")